# 奖励模型 (Reward Model) 训练

> RLHF 第二阶段：用人类偏好数据训练 Bradley-Terry 奖励模型。

## 背景
RLHF 需要 RM 对生成回答打分。RM 通常用 SFT 模型初始化，去掉 lm_head 换成一个标量头。
训练目标：让 chosen 回答得分 > rejected 回答得分。

## 损失函数（Bradley-Terry）
P(chosen > rejected) = sigma(r_chosen - r_rejected)
L = -log sigma(r_c - r_r)

## 架构
- base model (SFT checkpoint, 冻结或微调最后几层)
- value head: Linear(hidden_dim, 1)
- 输入：(prompt, chosen_response, rejected_response) 三元组

## 考察点
- Bradley-Terry 模型推导
- RM 与 PPO/DPO 中 reward 的对应关系
- RM over-optimization 问题


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RewardModel(nn.Module):
    """Reward Model: 对序列最后一 token 的 hidden 做线性投影输出标量奖励。"""
    def __init__(self, hidden_dim: int = 64) -> None:
        super().__init__()
        self.embed = nn.Embedding(1000, hidden_dim)
        self.rnn = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        h = self.embed(input_ids)
        _, h_n = self.rnn(h)
        reward = self.head(h_n[-1])
        return reward.squeeze(-1)

def bradley_terry_loss(chosen_rewards: torch.Tensor, rejected_rewards: torch.Tensor) -> torch.Tensor:
    """Bradley-Terry 偏好损失: -log(sigmoid(r_chosen - r_rejected))."""
    logits = chosen_rewards - rejected_rewards
    loss = -F.logsigmoid(logits).mean()
    return loss

# 验证
model = RewardModel(hidden_dim=16)
x = torch.randint(0, 1000, (2, 10))
r = model(x)
assert r.shape == (2,), f"reward shape: {r.shape}"
print(f"✅ RewardModel: 输出形状正确, rewards={r.tolist()}")

r_chosen = torch.tensor([2.0, 3.0])
r_rejected = torch.tensor([1.0, 0.5])
loss = bradley_terry_loss(r_chosen, r_rejected)
assert loss.item() > 0, "loss should be positive"
print(f"✅ BradleyTerry: loss={loss.item():.4f}")
